# Flavor Expansion Walkthrough

This notebook demonstrates the new flavor-index and class-member expansion features.

It covers:

1. explicit `IndexRole.FLAVOR` metadata,
2. generic fields with concrete class members,
3. compact vs expanded Feynman rules,
4. non-diagonal flavor tensors,
5. one-index diagonal tensors with `allow_summation=True`,
6. zero tensor components dropping expanded vertices,
7. gauge indices staying symbolic,
8. shared and independent flavor labels across field classes,
9. representative error messages.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S
from symbolica.community.spenso import Representation

from model import (
    COLOR_FUND_INDEX,
    Field,
    IndexRole,
    IndexType,
    Lagrangian,
    Model,
    Parameter,
    SPINOR_INDEX,
)

print("Python:", sys.executable)
print("Repository root:", ROOT)


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
Repository root: /Users/rems/Documents/MScThesis


In [2]:
def canon(expr):
    text = expr.expand().to_canonical_string() if hasattr(expr, "expand") else str(expr)
    replacements = {
        "python::{}::": "",
        "python::": "",
        "spenso::{symmetric,real}::": "",
        "spenso::{spenso::upper}::": "",
        "spenso::": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def show_signatures(title, rules, limit=None):
    print(title)
    keys = list(rules)
    print("count:", len(keys))
    shown = keys if limit is None else keys[:limit]
    for key in shown:
        print(" ", key)
    if limit is not None and len(keys) > limit:
        print("  ...")
    print()


def show_vertex(title, expr):
    print(title)
    print(canon(expr))
    print()


def show_error(title, fn):
    print(title)
    try:
        fn()
    except Exception as exc:
        print(type(exc).__name__ + ":", exc)
    print()


def generation_index(name, dimension, prefix="f"):
    return IndexType(
        name,
        Representation.cof(dimension),
        kind=name.lower(),
        dimension=dimension,
        role=IndexRole.FLAVOR,
        prefix=prefix,
    )


def dirac_field(name, *, indices=(), symbol=None, conjugate_symbol=None):
    if symbol is None:
        symbol = S(name)
    if conjugate_symbol is None:
        conjugate_symbol = S(f"{name}bar")
    return Field(
        name,
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=symbol,
        conjugate_symbol=conjugate_symbol,
        indices=indices,
    )


def scalar_field(name, *, symbol=None):
    if symbol is None:
        symbol = S(name)
    return Field(name, spin=0, self_conjugate=True, symbol=symbol)


def flavor_family(generic_name, member_names, generation, *, extra_indices=()):
    members = tuple(
        dirac_field(
            member_name,
            indices=extra_indices + (SPINOR_INDEX,),
            symbol=S(member_name),
            conjugate_symbol=S(f"{member_name}bar"),
        )
        for member_name in member_names
    )
    generic = Field(
        generic_name,
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S(generic_name.lower()),
        conjugate_symbol=S(f"{generic_name.lower()}bar"),
        indices=(generation,) + extra_indices + (SPINOR_INDEX,),
        flavor_index=generation,
        class_members=members,
    )
    return generic, members


## 1. Define a flavor index and a generic field class

The generic field carries the flavor slot. Each class member drops that slot.


In [3]:
generation = generation_index("Generation", 3)
psi, (e, mu, tau) = flavor_family("Psi", ("e", "mu", "tau"), generation)
phi = scalar_field("Phi")

print("generation.role:", generation.role)
print("generation.dimension:", generation.dimension)
print("psi.indices:", [index.name for index in psi.indices])
print("psi.flavor_index_slot():", psi.flavor_index_slot())
print("members:", [member.name for member in psi.class_members])
print("member indices:", {member.name: [index.name for index in member.indices] for member in psi.class_members})
print("mapping:", {k: psi.class_member_for(k).name for k in range(1, generation.dimension + 1)})


generation.role: IndexRole.FLAVOR
generation.dimension: 3
psi.indices: ['Generation', 'Spinor']
psi.flavor_index_slot(): 0
members: ['e', 'mu', 'tau']
member indices: {'e': ['Spinor'], 'mu': ['Spinor'], 'tau': ['Spinor']}
mapping: {1: 'e', 2: 'mu', 3: 'tau'}


## 2. Compact vs expanded diagonal interaction

The compact rule keeps the generic field and a flavor identity tensor. The expanded rule produces only the diagonal member vertices.


In [4]:
f = S("f")
diagonal = Lagrangian(S("g") * psi.bar(f) * psi(f) * phi)

compact_rules = diagonal.feynman_rules(
    flavor_expand=False,
    key_format="names",
    simplify=True,
    include_delta=True,
)
expanded_rules = diagonal.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)

show_signatures("compact signatures", compact_rules)
show_signatures("expanded signatures", expanded_rules)
show_vertex(
    "compact rule Psi.bar, Psi, Phi",
    diagonal.feynman_rule(psi.bar, psi, phi, simplify=True, include_delta=True, flavor_expand=False),
)
show_vertex(
    "expanded rule e.bar, e, Phi",
    diagonal.feynman_rule(e.bar, e, phi, simplify=True, include_delta=True, flavor_expand=True),
)
show_error(
    "off-diagonal e.bar, mu, Phi is absent",
    lambda: diagonal.feynman_rule(e.bar, mu, phi, simplify=True, include_delta=True, flavor_expand=True),
)


compact signatures
count: 1
  ('Psi.bar', 'Psi', 'Phi')

expanded signatures
count: 3
  ('e.bar', 'e', 'Phi')
  ('mu.bar', 'mu', 'Phi')
  ('tau.bar', 'tau', 'Phi')

compact rule Psi.bar, Psi, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*g*g(bis(4,i1),bis(4,i2))*g(cof(3,f1),cof(3,f2))

expanded rule e.bar, e, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*g*g(bis(4,i1),bis(4,i2))

off-diagonal e.bar, mu, Phi is absent
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - tau.bar, tau, Phi



## 3. Non-diagonal two-index flavor tensor

Independent flavor labels produce all member pairs.


In [5]:
f_mix = S("fMix")
g_mix = S("gMix")
Y = Parameter("Y", indices=(generation, generation))
mixed_model = Model(
    fields=(psi, phi),
    parameters=(Y,),
    lagrangian_decl=S("gY") * psi.bar(f_mix) * Y(f_mix, g_mix) * psi(g_mix) * phi,
)
mixed_L = mixed_model.lagrangian()
mixed_rules = mixed_L.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)

show_signatures("expanded signatures for Y[f, g]", mixed_rules, limit=9)
show_vertex(
    "e.bar, mu, Phi",
    mixed_L.feynman_rule(e.bar, mu, phi, simplify=True, include_delta=True, flavor_expand=True),
)
show_vertex(
    "mu.bar, e, Phi",
    mixed_L.feynman_rule(mu.bar, e, phi, simplify=True, include_delta=True, flavor_expand=True),
)
show_vertex(
    "tau.bar, tau, Phi",
    mixed_L.feynman_rule(tau.bar, tau, phi, simplify=True, include_delta=True, flavor_expand=True),
)


expanded signatures for Y[f, g]
count: 9
  ('e.bar', 'e', 'Phi')
  ('e.bar', 'mu', 'Phi')
  ('e.bar', 'tau', 'Phi')
  ('mu.bar', 'e', 'Phi')
  ('mu.bar', 'mu', 'Phi')
  ('mu.bar', 'tau', 'Phi')
  ('tau.bar', 'e', 'Phi')
  ('tau.bar', 'mu', 'Phi')
  ('tau.bar', 'tau', 'Phi')

e.bar, mu, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*Y(1,2)*gY*g(bis(4,i1),bis(4,i2))

mu.bar, e, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*Y(2,1)*gY*g(bis(4,i1),bis(4,i2))

tau.bar, tau, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*Y(3,3)*gY*g(bis(4,i1),bis(4,i2))



## 4. One-index diagonal tensor and `allow_summation=True`

The shorthand `y[f] Psi.bar[f] Psi[f] Phi` is rejected unless the parameter explicitly opts in.


In [6]:
y_bad = Parameter("yBad", indices=(generation,))
bad_model = Model(
    fields=(psi, phi),
    parameters=(y_bad,),
    lagrangian_decl=y_bad(f) * psi.bar(f) * psi(f) * phi,
)
show_error(
    "without allow_summation",
    lambda: bad_model.lagrangian().vertex_signatures(flavor_expand=True),
)

y_diag = Parameter("yDiag", indices=(generation,), allow_summation=True)
diag_model = Model(
    fields=(psi, phi),
    parameters=(y_diag,),
    lagrangian_decl=y_diag(f) * psi.bar(f) * psi(f) * phi,
)
diag_L = diag_model.lagrangian()
diag_rules = diag_L.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)

show_signatures("diagonal y[f] expansion", diag_rules)
show_vertex(
    "mu.bar, mu, Phi",
    diag_L.feynman_rule(mu.bar, mu, phi, simplify=True, include_delta=True, flavor_expand=True),
)


without allow_summation
ValueError: Parameter 'yBad' uses flavor label 'f' more than twice in one term. Mark the parameter with allow_summation=True to use this diagonal shorthand.

diagonal y[f] expansion
count: 3
  ('e.bar', 'e', 'Phi')
  ('mu.bar', 'mu', 'Phi')
  ('tau.bar', 'tau', 'Phi')

mu.bar, mu, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*yDiag(2)*g(bis(4,i1),bis(4,i2))



## 5. Zero tensor components are dropped

Here all off-diagonal `Ydiag(i, j)` entries are defined as zero, so only the three diagonal vertices survive.


In [7]:
Ydiag = Parameter(
    "Ydiag",
    indices=(generation, generation),
    components={
        (1, 2): 0,
        (1, 3): 0,
        (2, 1): 0,
        (2, 3): 0,
        (3, 1): 0,
        (3, 2): 0,
    },
)
zero_model = Model(
    fields=(psi, phi),
    parameters=(Ydiag,),
    lagrangian_decl=S("gDiag") * psi.bar(f_mix) * Ydiag(f_mix, g_mix) * psi(g_mix) * phi,
)
zero_L = zero_model.lagrangian()
zero_rules = zero_L.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)

show_signatures("only diagonal signatures remain", zero_rules)
show_error(
    "e.bar, mu, Phi was dropped",
    lambda: zero_L.feynman_rule(e.bar, mu, phi, simplify=True, include_delta=True, flavor_expand=True),
)


only diagonal signatures remain
count: 3
  ('e.bar', 'e', 'Phi')
  ('mu.bar', 'mu', 'Phi')
  ('tau.bar', 'tau', 'Phi')

e.bar, mu, Phi was dropped
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - tau.bar, tau, Phi



## 6. Gauge indices are not flavor-expanded

A quark class may carry `(generation, color, spinor)`. Expansion removes only the generation slot.


In [8]:
generation_q = generation_index("GenerationQ", 2, prefix="fq")
q, (d, u) = flavor_family("q", ("d", "u"), generation_q, extra_indices=(COLOR_FUND_INDEX,))
phi_q = scalar_field("PhiQ")
fq = S("fq")
c = S("c")
q_L = Lagrangian(S("gQ") * q.bar(fq, c) * q(fq, c) * phi_q)
expanded_terms = q_L._expanded_terms(flavor_expand=True)

for term in expanded_terms:
    print(term.fields[0].field.name, term.fields[0].labels, term.fields[1].field.name, term.fields[1].labels)
print()
show_vertex(
    "d.bar, d, PhiQ still carries a color identity tensor",
    q_L.feynman_rule(d.bar, d, phi_q, simplify=True, include_delta=True, flavor_expand=True),
)


d {'color_fund': c, 'spinor': i_decl_1} d {'color_fund': c, 'spinor': i_decl_1}
u {'color_fund': c, 'spinor': i_decl_1} u {'color_fund': c, 'spinor': i_decl_1}

d.bar, d, PhiQ still carries a color identity tensor
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*gQ*g(bis(4,i1),bis(4,i2))*g(cof(3,c1),cof(3,c2))



## 7. Shared and independent flavor labels across classes

The same flavor label can synchronize two different classes. Different labels remain independent.


In [9]:
L_field, (eL, muL, tauL) = flavor_family("L", ("eL", "muL", "tauL"), generation)
R_field, (eR, muR, tauR) = flavor_family("R", ("eR", "muR", "tauR"), generation)
yLR = Parameter("yLR", indices=(generation,), allow_summation=True)
shared_model = Model(
    fields=(L_field, R_field, phi),
    parameters=(yLR,),
    lagrangian_decl=yLR(f) * L_field.bar(f) * R_field(f) * phi,
)
shared_L = shared_model.lagrangian()
shared_rules = shared_L.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)
show_signatures("shared flavor label across L and R", shared_rules)
show_vertex(
    "tauL.bar, tauR, Phi",
    shared_L.feynman_rule(tauL.bar, tauR, phi, simplify=True, include_delta=True, flavor_expand=True),
)

U_field, (u_phys, c_phys, t_phys) = flavor_family("U", ("u", "c", "t"), generation)
D_field, (d_phys, s_phys, b_phys) = flavor_family("D", ("dQ", "sQ", "bQ"), generation)
W = scalar_field("W")
V = Parameter("V", indices=(generation, generation))
independent_model = Model(
    fields=(U_field, D_field, W),
    parameters=(V,),
    lagrangian_decl=U_field.bar(f_mix) * V(f_mix, g_mix) * D_field(g_mix) * W,
)
independent_L = independent_model.lagrangian()
independent_rules = independent_L.feynman_rules(
    flavor_expand=True,
    key_format="names",
    simplify=True,
    include_delta=True,
)
show_signatures("independent U[f] and D[g] labels", independent_rules, limit=9)
show_vertex(
    "u.bar, sQ, W",
    independent_L.feynman_rule(u_phys.bar, s_phys, W, simplify=True, include_delta=True, flavor_expand=True),
)
show_vertex(
    "t.bar, bQ, W",
    independent_L.feynman_rule(t_phys.bar, b_phys, W, simplify=True, include_delta=True, flavor_expand=True),
)


shared flavor label across L and R
count: 3
  ('eL.bar', 'eR', 'Phi')
  ('muL.bar', 'muR', 'Phi')
  ('tauL.bar', 'tauR', 'Phi')

tauL.bar, tauR, Phi
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*yLR(3)*g(bis(4,i1),bis(4,i2))

independent U[f] and D[g] labels
count: 9
  ('u.bar', 'dQ', 'W')
  ('u.bar', 'sQ', 'W')
  ('u.bar', 'bQ', 'W')
  ('c.bar', 'dQ', 'W')
  ('c.bar', 'sQ', 'W')
  ('c.bar', 'bQ', 'W')
  ('t.bar', 'dQ', 'W')
  ('t.bar', 'sQ', 'W')
  ('t.bar', 'bQ', 'W')

u.bar, sQ, W
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*V(1,2)*g(bis(4,i1),bis(4,i2))

t.bar, bQ, W
(2*𝜋)^d*1𝑖*Delta(q1+q2+q3)*V(3,3)*g(bis(4,i1),bis(4,i2))



## 8. Representative error messages

The new checks try to fail early and clearly when declarations are inconsistent.


In [10]:
show_error(
    "wrong member count",
    lambda: Field(
        "BadPsi",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("badpsi"),
        conjugate_symbol=S("badpsibar"),
        indices=(generation, SPINOR_INDEX),
        flavor_index=generation,
        class_members=(e, mu),
    ),
)

psi_no_members = Field(
    "PsiNoMembers",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi_no_members"),
    conjugate_symbol=S("psi_no_members_bar"),
    indices=(generation, SPINOR_INDEX),
    flavor_index=generation,
)
missing_members_L = Lagrangian(S("gMissing") * psi_no_members.bar(f) * psi_no_members(f) * phi)
show_error(
    "flavor expansion requested but no class members are defined",
    lambda: missing_members_L.vertex_signatures(flavor_expand=True),
)

plain_index = IndexType(
    "PlainGeneration",
    Representation.cof(3),
    kind="plain_generation",
    dimension=3,
    prefix="p",
)
show_error(
    "flavor_index must use role=IndexRole.FLAVOR",
    lambda: Field(
        "WrongRole",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("wrongrole"),
        conjugate_symbol=S("wrongrolebar"),
        indices=(plain_index, SPINOR_INDEX),
        flavor_index=plain_index,
        class_members=(e, mu, tau),
    ),
)


wrong member count
ValueError: Field 'BadPsi' declares 2 class member(s), but flavor index 'Generation' has dimension 3.

flavor expansion requested but no class members are defined
ValueError: Cannot flavor-expand field 'PsiNoMembers' over index 'Generation' because no class members are defined.

flavor_index must use role=IndexRole.FLAVOR
ValueError: Field 'WrongRole' declares flavor_index='PlainGeneration', but that index type is not marked with role=IndexRole.FLAVOR.

